## tl;dr

This notebook audits whether GenForms can attribute SEO clicks to create, publish, non-test submission, upgrade intent, and qualified lead. The 2026-07-03 audit found that GSC and product fact tables are usable separately, but Form records do not persist SEO acquisition context, outcome events lose visitor/session identity, and qualified lead is not instrumented. SEO publish/submit conversion rates are therefore not decision-safe.

## Context & Methods

Reader: Growth/Product/Codex. Decision: whether Production Loop batches may Scale, Tune, Hold, or Stop. Sources are production Supabase tables `growth_events`, `forms`, `form_submissions`, and `growth_metric_snapshots`. The notebook reads only bounded non-PII fields.

### Key Assumptions

- `forms` controls create/current publish facts.
- `form_submissions` with `is_test=false` controls real submission facts.
- GSC controls organic clicks.
- Growth events are diagnostic and must not be summed with server facts.
- Admin paths and events explicitly marked `is_dev=true` are excluded; this still does not remove every internal server event.

In [1]:
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import json
import os
import pandas as pd

ROOT = Path.cwd()
if ROOT.name != 'AIFactory':
    candidates = [ROOT, *ROOT.parents]
    ROOT = next((p for p in candidates if (p / 'Code' / '.env.local').exists()), ROOT)
START_7D = '2026-06-27T00:00:00.000Z'
START_28D = '2026-06-06T00:00:00.000Z'
END = '2026-07-03T23:59:59.999Z'

def load_env_file(path):
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        value = value.strip().strip('\"').strip("'")
        os.environ.setdefault(key.strip(), value)

load_env_file(ROOT / 'Code' / '.env.local')
SUPABASE_URL = os.environ['SUPABASE_URL'].rstrip('/')
SUPABASE_KEY = os.environ.get('SUPABASE_SERVICE_ROLE_KEY') or os.environ['SUPABASE_ANON_KEY']
print({'workspace': str(ROOT), 'window_7d': [START_7D, END], 'window_28d': [START_28D, END]})

{'workspace': '/Users/mike/Documents/AIFactory', 'window_7d': ['2026-06-27T00:00:00.000Z', '2026-07-03T23:59:59.999Z'], 'window_28d': ['2026-06-06T00:00:00.000Z', '2026-07-03T23:59:59.999Z']}


In [2]:
def supabase_rows(table, select, start, end, page_size=1000):
    rows = []
    offset = 0
    while True:
        params = {
            'select': select,
            'created_at': [f'gte.{start}', f'lte.{end}'],
            'order': 'created_at.asc',
            'offset': offset,
            'limit': page_size,
        }
        query_parts = [f'select={select}', f'created_at=gte.{start}', f'created_at=lte.{end}', 'order=created_at.asc', f'offset={offset}', f'limit={page_size}']
        url = f"{SUPABASE_URL}/rest/v1/{table}?" + '&'.join(query_parts)
        request = Request(url, headers={'apikey': SUPABASE_KEY, 'Authorization': f'Bearer {SUPABASE_KEY}'})
        with urlopen(request, timeout=30) as response:
            batch = json.loads(response.read().decode('utf-8'))
        rows.extend(batch)
        if len(batch) < page_size:
            break
        offset += page_size
    return rows

def load_window(start):
    events = supabase_rows('growth_events', 'event_name,visitor_id,session_id,path,source,template_id,form_uuid,metadata_json,created_at', start, END)
    forms = supabase_rows('forms', 'uuid,status,generation_meta_json,created_at', start, END)
    submissions = supabase_rows('form_submissions', 'uuid,form_uuid,is_test,status,created_at', start, END)
    return events, forms, submissions

events_7d, forms_7d, submissions_7d = load_window(START_7D)
events_28d, forms_28d, submissions_28d = load_window(START_28D)
print({'rows_7d': [len(events_7d), len(forms_7d), len(submissions_7d)], 'rows_28d': [len(events_28d), len(forms_28d), len(submissions_28d)]})

{'rows_7d': [762, 2, 2], 'rows_28d': [2868, 2, 2]}


## Data

The profile below intentionally reports aggregates only. It does not print emails, prompts, answers, form titles, or submission payloads.

In [3]:
FUNNEL_EVENTS = ['landing_viewed', 'template_used', 'forms_new_view', 'ai_generate_submitted', 'form_created', 'form_published', 'publish_succeeded', 'test_submission_completed', 'first_result_viewed', 'activation_completed', 'public_form_submitted', 'paywall_clicked', 'checkout_started', 'purchase_completed']
CONTENT_EVENTS = {'template_used', 'forms_new_view', 'ai_generate_submitted'}

def clean_events(rows):
    return [row for row in rows if not (row.get('metadata_json') or {}).get('is_dev') and '/admin' not in str(row.get('path') or '')]

def profile(label, events, forms, submissions):
    clean = clean_events(events)
    counts = {name: sum(row['event_name'] == name for row in clean) for name in FUNNEL_EVENTS}
    content = [row for row in clean if row['event_name'] in CONTENT_EVENTS]
    intent_usable = sum((row.get('metadata_json') or {}).get('intent') not in (None, '', 'unspecified') for row in content)
    outcome_identity = {}
    for name in ['form_created', 'form_published', 'public_form_submitted']:
        subset = [row for row in clean if row['event_name'] == name]
        outcome_identity[name] = {
            'rows': len(subset),
            'blank_visitor': sum(not row.get('visitor_id') for row in subset),
            'blank_session': sum(not row.get('session_id') for row in subset),
        }
    return {
        'window': label,
        'raw_event_rows': len(events),
        'clean_event_rows': len(clean),
        'excluded_share_pct': round((1 - len(clean) / len(events)) * 100, 1) if events else 0,
        'forms_created': len(forms),
        'forms_currently_published': sum(row.get('status') == 'published' for row in forms),
        'non_test_submissions': sum(not row.get('is_test') for row in submissions),
        'test_submissions': sum(bool(row.get('is_test')) for row in submissions),
        'funnel_event_rows': counts,
        'content_events': len(content),
        'usable_intent_events': intent_usable,
        'outcome_identity': outcome_identity,
    }

profiles = [profile('7d', events_7d, forms_7d, submissions_7d), profile('28d', events_28d, forms_28d, submissions_28d)]
pd.DataFrame([{k:v for k,v in row.items() if k not in ('funnel_event_rows','outcome_identity')} for row in profiles])

,window,raw_event_rows,clean_event_rows,excluded_share_pct,forms_created,forms_currently_published,non_test_submissions,test_submissions,content_events,usable_intent_events
0,7d,762,262,65.6,2,2,1,1,5,0
1,28d,2868,1539,46.3,2,2,1,1,63,0


## Results

In [4]:
results = {
    'profiles': profiles,
    'forms_with_persisted_attribution_7d': sum(bool((row.get('generation_meta_json') or {}).get('attribution')) for row in forms_7d),
    'forms_with_persisted_attribution_28d': sum(bool((row.get('generation_meta_json') or {}).get('attribution')) for row in forms_28d),
    'qualified_lead_instrumented': False,
}
print(json.dumps(results, indent=2))

{
  "profiles": [
    {
      "window": "7d",
      "raw_event_rows": 762,
      "clean_event_rows": 262,
      "excluded_share_pct": 65.6,
      "forms_created": 2,
      "forms_currently_published": 2,
      "non_test_submissions": 1,
      "test_submissions": 1,
      "funnel_event_rows": {
        "landing_viewed": 60,
        "template_used": 2,
        "forms_new_view": 2,
        "ai_generate_submitted": 1,
        "form_created": 1,
        "form_published": 0,
        "publish_succeeded": 0,
        "test_submission_completed": 1,
        "first_result_viewed": 0,
        "activation_completed": 0,
        "public_form_submitted": 1,
        "paywall_clicked": 0,
        "checkout_started": 0,
        "purchase_completed": 0
      },
      "content_events": 5,
      "usable_intent_events": 0,
      "outcome_identity": {
        "form_created": {
          "rows": 1,
          "blank_visitor": 1,
          "blank_session": 1
        },
        "form_published": {
          "row

## Takeaways

1. Use GSC for search clicks and page/query ownership.
2. Use Form and non-test Submission tables for business facts.
3. Do not calculate SEO publish or submission conversion until acquisition context is persisted on Form.
4. Treat Growth/GA4 events as diagnostic until event identity, session expiry, internal filtering, and content intent completeness are repaired.
5. Keep qualified lead as N/A, not zero, until a reviewed business outcome source exists.